# EAIOS M3 · acceptance run (Colab T4, loopback only)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all** (about 45–75 minutes, most of it installs and model downloads).

The model gateway (Qwen3-8B-AWQ, BGE-M3, the development checker, Docling) and the application (PostgreSQL 17, Redis, Qdrant v1.15.4, API) run in this runtime and talk over `127.0.0.1` only. No tunnel is opened. The last cell downloads a credential-free evidence zip. Details: `ml/colab-m3/README.md`.

In [ ]:
# @title 1 · Clone the repository at the branch under review
BRANCH = "feature/m3-authorized-document-answers"  # @param {type:"string"}
import os, subprocess
REPO = "/content/Enterprise-AI-OS"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS", REPO], check=True)
else:
    subprocess.run(f"git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q FETCH_HEAD", shell=True, check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# @title 2 · Install the pinned vLLM environment and the M1 Docling environment
!python /content/Enterprise-AI-OS/ml/colab-m3/run_m3_colab.py vllm-env
!python /content/Enterprise-AI-OS/ml/colab-m3/run_m3_colab.py docling-env

In [ ]:
# @title 3 · Start the model gateway with Docling (127.0.0.1:8100)
import os, secrets
os.environ.setdefault("GATEWAY_TOKEN", secrets.token_urlsafe(32))  # never printed; stays in this runtime
os.environ["DOCLING_PYTHON"] = "/content/venv-docling/bin/python"
%run /content/Enterprise-AI-OS/ml/model-gateway/colab_launch.py

In [ ]:
# @title 4 · Services, API, demo capture, retrieval and relevance measurements, parser checks, fallback, bundle
!python /content/Enterprise-AI-OS/ml/colab-m3/run_m3_colab.py all

In [ ]:
# @title 5 · Download the evidence zip
import glob
from google.colab import files
latest = sorted(glob.glob("/content/m3-evidence-*.zip"))[-1]
print(latest)
files.download(latest)